# 6. Diversity Analysis (Alpha & Beta)
## Import data & packages

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import qiime2 as q2
from qiime2 import Visualization
import seaborn as sns
from scipy.stats import shapiro, kruskal, f_oneway
  
%matplotlib inline

In [ ]:
# 2 - Set working directory
# The working directory should normally default to the 'scripts' folder. 
# If it doesn't, set it manually using the command below.
# os.chdir("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts")  # Adjust this path to match your folder structure.

# Verify that your working directory is the 'scripts' folder inside the main project directory (.../MicrobiomeAnalysis_TummyTribe/scripts)
cwd = os.getcwd()
if not cwd.endswith("MicrobiomeAnalysis_TummyTribe/scripts"):
    print("WARNING: The working directory is not set to the 'scripts' folder inside 'MicrobiomeAnalysis_TummyTribe'!")
    print("Current working directory:", cwd)
    print("Please set it manually using os.chdir().")
else:
    print(f"Working directory is correctly set to the 'scripts' folder (\"{cwd}\").")

In [ ]:
# 3 - Data directories
raw_data_dir = "../data/raw"
denoising_data_dir = "../data/processed/denoising"
DA_data_dir = "../data/processed/differentialanalysis"

# 1. Data exploration: look at feature table

In [ ]:
data = q2.Artifact.load(f'{processed_data_dir}/dada2_table.qza').view(pd.DataFrame)

In [ ]:
data.head()

Plot the distribution of 15 samples to see if data is normally distributed seaborn's violin plot:

In [ ]:
n = 15

# draw n ASVs out of the original DataFrame
data_samp = data.sample(n=n, axis=1, random_state=234)

# create a new DataFrame with three columns (sample, ASV, abundance)
col_names = {'level_0': 'sample', 'level_1': 'asv', 0: 'count'}
data_plot = data_samp.stack().reset_index().rename(columns=col_names)

# make sure the shape is correct (no. of rows should be equal to the size of our random 
# sample multiplied by number of samples)
assert data_plot.shape[0] == n * data_samp.shape[0], 'The new DataFrame has an incorrect no. of rows.'

In [ ]:
with sns.axes_style('white'), sns.color_palette('Set1'):
    fig, ax = plt.subplots()
    fig.set_size_inches(15, 7.5)
    
    sns.violinplot(data=data_plot, x='asv', y='count', ax=ax)
    sns.despine(left=True)
    
    # adjust tick labels and axes titles
    ax.tick_params(axis='x', rotation=90, labelsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.set_xlabel('ASV', fontsize=14)
    ax.set_ylabel('Count', fontsize=14)

From this we can suspect that the data is not normally distributed. We can additionally check this by statistically testing for normality. 

In [ ]:
alpha = 0.05
results = {}

# iterate through rows (samples) and test each of them for normality
for asv_name, asv_values in data.items():
    stat, p = shapiro(asv_values)
    results[asv_name] = p

# convert test results into a DataFrame
results_df = pd.DataFrame(data=results.values(), index=results.keys(), columns=['p'])

# add a new column with a descriptive test result
results_df['is_normal'] = results_df['p'] > alpha

In [ ]:
results_df

In [ ]:
print('Number of ASVs with normal distribution:', results_df['is_normal'].sum())

# 2. ANCOM – Delivery Mode

In [ ]:
! qiime feature-table filter-features \
    --i-table $denoising_data_dir/dada2_table.qza \
    --p-min-frequency 25 \
    --p-min-samples 4 \
    --o-filtered-table $DA_data_dir/table_abund.qza

In [ ]:
! qiime feature-table filter-samples \
    --i-table $DA_data_dir/table_abund.qza \
    --m-metadata-file $raw_data_dir/metadata.tsv \
    --p-where "[delivery_mode]='vaginal' or [delivery_mode]='cesarean'" \
    --o-filtered-table $DA_data_dir/delivery_mode/table_abund_deliverymode.qza

In [ ]:
# Run ANCOM-BC
! qiime composition ancombc \
    --i-table $DA_data_dir/delivery_mode/table_abund_deliverymode.qza \
    --m-metadata-file $raw_data_dir/metadata.tsv \
    --p-formula delivery_mode \
    --o-differentials $DA_data_dir/delivery_mode/ancombc_deliverymode_differentials.qza

# Generate a barplot of differentially abundant taxa between environments
! qiime composition da-barplot \
    --i-data $DA_data_dir/delivery_mode/ancombc_deliverymode_differentials.qza \
    --o-visualization $DA_data_dir/delivery_mode/ancombc_deliverymode_da_barplot.qzv

# Generate a table of these same values for all taxa
! qiime composition tabulate \
    --i-data $DA_data_dir/delivery_mode/ancombc_deliverymode_differentials.qza \
    --o-visualization $DA_data_dir/delivery_mode/ancombc_deliverymode_results.qzv

In [ ]:
Visualization.load(f"{DA_data_dir}/delivery_mode/ancombc_deliverymode_da_barplot.qzv")

In [ ]:
Visualization.load(f"{DA_data_dir}/delivery_mode/ancombc_deliverymode_results.qzv")

# 3. ANCOM – Diet Milk

In [ ]:
! qiime feature-table filter-samples \
    --i-table $DA_data_dir/table_abund.qza \
    --m-metadata-file $raw_data_dir/metadata.tsv \
    --p-where "[diet_milk]='bd' or [diet_milk]='fd'" \
    --o-filtered-table $DA_data_dir/diet/table_abund_diet.qza

In [ ]:
# Run ANCOM-BC
! qiime composition ancombc \
    --i-table $DA_data_dir/diet/table_abund_diet.qza \
    --m-metadata-file $raw_data_dir/metadata.tsv \
    --p-formula diet_milk \
    --o-differentials $DA_data_dir/diet/ancombc_diet_differentials.qza

# Generate a barplot of differentially abundant taxa between environments
! qiime composition da-barplot \
    --i-data $DA_data_dir/diet/ancombc_diet_differentials.qza \
    --o-visualization $DA_data_dir/diet/ancombc_deliverymode_da_barplot.qzv

# Generate a table of these same values for all taxa
! qiime composition tabulate \
    --i-data $DA_data_dir/diet/ancombc_deliverymode_differentials.qza \
    --o-visualization $DA_data_dir/diet/ancombc_deliverymode_results.qzv

In [ ]:
Visualization.load(f"{DA_data_dir}/diet/ancombc_diet_da_barplot.qzv")

In [ ]:
Visualization.load(f"{DA_data_dir}/diet/ancombc_diet_results.qzv")